# Lab 2: Problems and Hints

Use linear regression to study house prices in the Ames dataset. In Problem 1, estimate how much fitted coefficients vary. In Problem 2, choose regularization penalties using cross-validation.

**How to work:** Run the setup cells, then complete one section at a time. In a starter cell, replace `____` and remove the leading `#` from the lines you complete. A line beginning with `#` is a Python comment, so unfinished templates do not run. Write short explanations in the answer cells.

Keep `data/ames_train.csv` beside this notebook. The data and feature definitions are also used in **Lab2Master.ipynb**. This notebook runs on its own.


## Setup

If an import fails, run `%pip install numpy pandas matplotlib scipy scikit-learn` in a new cell, then restart the kernel.

Three useful Python details:

- `df[["column"]]` gives a two-dimensional table, suitable for a model's predictors.
- `df["column"]` gives a one-dimensional Series, suitable for the response.
- `.iloc[positions]` selects rows by position, starting at zero.

In Colab, upload `ames_train.csv` using the Files panel. The loading cell also supports downloading the course CSV.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t
from IPython.display import display

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

SEED = 7243
P2_SEED = 255
BOOTSTRAP_SEED = 255
plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": 0.2})


In [2]:
DATA_URL = "https://raw.githubusercontent.com/tipthederiver/Math-7243-2020/master/Datasets/Ames/train.csv"
data_path = Path("data/ames_train.csv")
if not data_path.exists() and Path("ames_train.csv").exists():
    data_path = Path("ames_train.csv")

if data_path.exists():
    ames = pd.read_csv(data_path)
else:
    try:
        ames = pd.read_csv(DATA_URL)
    except Exception as exc:
        raise RuntimeError("Upload ames_train.csv or place it in the data folder.") from exc
    data_path.parent.mkdir(parents=True, exist_ok=True)
    ames.to_csv(data_path, index=False)

# Keep houses with at most 4,000 square feet of above-ground living area.
data = ames.loc[ames["GrLivArea"] <= 4000].copy()
assert data[["SalePrice", "1stFlrSF"]].notna().all().all()
print("Houses available:", len(data))
display(data[["1stFlrSF", "SalePrice"]].head())


Houses available: 1456


,1stFlrSF,SalePrice
0,856,208500
1,1262,181500
2,920,223500
3,961,140000
4,1145,250000


## Problem 1: Confidence intervals by bootstrapping

Fit the model

$$\mathrm{SalePrice}_i=\beta_0+\beta_1\,\mathrm{1stFlrSF}_i+\varepsilon_i.$$

`1stFlrSF` measures first-floor area in square feet. Use the training portion below for every calculation in this problem.

1. Fit the regression once using all training rows.
2. Draw **1,000 bootstrap samples**. Each sample contains **$n$ training rows, with replacement**, where $n$ is the training-set size. Fit a regression to each sample and save its intercept and slope.
3. Plot a histogram for each coefficient. Find a 95% bootstrap interval using the **middle 950** sorted estimates.
4. Compute analytical 95% intervals using the lecture's covariance formula and Student-$t$ distribution.
5. Compare the interval centers and widths.

**Remember:** $n$ counts houses in each sample; $B=1000$ counts repeated fits.


In [3]:
p1_train, p1_reserved = train_test_split(data, test_size=0.20, random_state=SEED)
X1 = p1_train[["1stFlrSF"]]
y1 = p1_train["SalePrice"]
n = len(p1_train)
B = 1000
print("Training rows:", n)
print("X1 shape:", X1.shape, "| y1 shape:", y1.shape)


Training rows: 1164
X1 shape: (1164, 1) | y1 shape: (1164,)


### 1A. Fit and resample

Lecture pp. **15–17** explain that fitted coefficients vary when the sample changes. The bootstrap approximates this variation using the observed data.

Sample **complete rows**: use the same positions for the area and price. Sampling the two columns separately breaks their relationship.

`rng.integers(0, n, size=n)` draws $n$ row positions with replacement. Create `rng` before the loop so each repetition receives a different sample. The fixed seed makes your work reproducible.

After fitting, `model.intercept_` gives the intercept and `model.coef_[0]` gives the slope. Store one of each at position `b` in the arrays.


In [4]:
# p1_model = LinearRegression()
# p1_model.fit(____, ____)

# rng = np.random.default_rng(BOOTSTRAP_SEED)
# beta0 = np.zeros(B)
# beta1 = np.zeros(B)
# for b in range(B):
#     positions = rng.integers(0, n, size=n)
#     X_boot = X1.iloc[____]
#     y_boot = y1.iloc[____]
#     model = LinearRegression()
#     model.fit(____, ____)
#     beta0[b] = ____
#     beta1[b] = ____

# Check that each array contains B estimates:
# print(beta0.shape, beta1.shape)


### 1B. Plot and find the middle 950

The intercept is measured in **dollars**; the slope is measured in **dollars per square foot**. Use separate histograms and label both axes.

For each coefficient:

1. Sort the 1,000 estimates with `np.sort`.
2. Remove the smallest 25 and largest 25.
3. Report the first and last remaining values as the endpoints.

Python slices include the starting position and exclude the stopping position: `values[25:975]` contains 950 values. The interval width is `upper - lower`. Keep this same convention for both coefficients.


In [5]:
# fig, axes = plt.subplots(1, 2, figsize=(10, 4))
# axes[0].hist(____, bins=30, edgecolor="white")
# axes[1].hist(____, bins=30, edgecolor="white")
# axes[0].set(xlabel="Intercept (dollars)", ylabel="Count")
# axes[1].set(xlabel="Slope (dollars per sq ft)", ylabel="Count")
# plt.tight_layout()
# plt.show()

# sorted_beta0 = np.sort(____)
# middle_beta0 = sorted_beta0[25:975]
# bootstrap_ci0 = (middle_beta0[0], middle_beta0[-1])
# Repeat these three lines for beta1 to create bootstrap_ci1.


### 1C. Use the lecture's analytical interval

Use **all training rows**, rather than a bootstrap sample, for this calculation. The lecture writes coefficients as $\theta$; here we use $\beta$ for the same quantities.

**Design matrix, pp. 15–17.** Add an intercept column:

$$A=\begin{bmatrix}1&x_1\\\vdots&\vdots\\1&x_n\end{bmatrix},\qquad
\hat\beta=\begin{bmatrix}\hat\beta_0\\\hat\beta_1\end{bmatrix},\qquad
\hat y=A\hat\beta.$$

The covariance formula is $\operatorname{Cov}(\hat\beta)=\sigma^2(A^TA)^{-1}$.

**Estimate the error variance, p. 20.** There is one predictor and an intercept, so the residual degrees of freedom are $n-2$:

$$r=y-\hat y,\qquad \mathrm{RSS}=r^Tr,\qquad s^2=\frac{\mathrm{RSS}}{n-2}.$$

**Find standard errors.** Form $\widehat C=s^2(A^TA)^{-1}$, then take square roots of its diagonal:

$$\operatorname{SE}(\hat\beta_j)=\sqrt{\widehat C_{jj}}.$$

`np.diag(C_hat)` extracts the diagonal. The entries are variances, so remember the square root.

**Build intervals, p. 22.** For $j=0,1$,

$$\hat\beta_j\pm t_{0.975,n-2}\operatorname{SE}(\hat\beta_j).$$

Use `t.ppf(0.975, df=n-2)` for the positive critical value. Page 21 gives the known-variance interval; here the variance is estimated. Exact $t$ coverage assumes a correct linear model with independent normal errors of constant variance.


In [6]:
# x = X1["1stFlrSF"].to_numpy()
# observed = y1.to_numpy()
# A = np.column_stack([np.ones(n), x])
# beta_hat = np.array([p1_model.intercept_, p1_model.coef_[0]])
# fitted = A @ ____
# residual = observed - ____
# RSS = residual @ residual
# s_squared = RSS / ____

# This solves for the inverse in the covariance formula.
# G = np.linalg.solve(A.T @ A, np.eye(2))
# C_hat = ____ * G
# standard_errors = np.sqrt(____)
# critical = t.ppf(____, df=____)
# lower = beta_hat - critical * ____
# upper = beta_hat + critical * ____

# Compare both coefficients in a table with these columns:
# coefficient | estimate | bootstrap lower | bootstrap upper
#             | bootstrap width | analytical lower | analytical upper | analytical width


### 1D. Compare the results

Write four or five sentences using your table and histograms:

- Are the two interval methods similar? Compare a center and a width numerically.
- Are the bootstrap distributions roughly symmetric?
- What does the slope interval suggest about the association between floor area and price?
- Why is a coefficient interval different from an interval for an individual house price?

Differences can reflect skewness, influential observations, unequal error variance, or variation from a finite number of resamples. A 95% confidence procedure aims to cover the fixed population coefficient in 95% of repeated samples under its assumptions. An association alone does not establish causation.


**My Problem 1 comparison:**

_Write your answer here._


## Problem 2: Ridge and lasso regression

Predict `SalePrice` using the 12 features below. Use an **80%/20% train/test split with seed 255** and **five-fold cross-validation** within the training set.

For **ridge** and **lasso**:

1. Search a grid of penalties and report the chosen value and mean validation RMSE.
2. Plot validation RMSE against the penalty.
3. Explain your choice in two or three sentences. For lasso, also count the nonzero slopes.
4. After choosing the penalties, report test RMSE, MAE, and $R^2$.

**Bonus:** Tune elastic net's two parameters using the same folds.

Lecture pp. **26–27** explain why training error alone is insufficient for judging prediction. Pages **34–36** connect regularization to bias and variance: a biased estimator can sometimes predict better than OLS, even when OLS is BLUE.


In [7]:
feature_names = [
    "GrLivArea", "OverallQual", "GarageCars", "TotalBsmtSF",
    "YearBuilt", "LotArea", "LotFrontage", "MasVnrArea",
    "FullBath", "TotRmsAbvGrd", "Fireplaces", "OverallCond",
]
X2 = data[feature_names]
y2 = data["SalePrice"]
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.20, random_state=P2_SEED
)
cv = KFold(n_splits=5, shuffle=True, random_state=P2_SEED)
print("Training:", X2_train.shape, "| Test:", X2_test.shape)


Training: (1164, 12) | Test: (292, 12)


### 2A. Prepare a pipeline

A pipeline keeps three steps together:

1. Fill missing values using training medians.
2. Standardize predictors using training means and standard deviations.
3. Fit the regression model.

Pass the whole pipeline to the search. Each fold then learns preprocessing from its fitting rows only. This prevents validation data from influencing the fit. The test data are transformed using the fitted pipeline later.

`OverallQual` and `OverallCond` are ordered scores treated as numeric. For all models, the intercept is fitted separately and is not penalized.


In [8]:
def regression_pipeline(model):
    return Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("model", model),
    ])

ridge_pipeline = regression_pipeline(Ridge(solver="svd"))
lasso_pipeline = regression_pipeline(Lasso(max_iter=50000, tol=1e-5))

# Starting grids: use training cross-validation to refine them if needed.
ridge_grid = {"model__alpha": np.logspace(-2, 6, 25)}
lasso_grid = {"model__alpha": np.logspace(0, 4.5, 25)}


### 2B. Search for a penalty

With standardized predictors $Z$, scikit-learn uses

$$\text{Ridge: }\|y-b\mathbf1-Z\beta\|_2^2+\alpha\|\beta\|_2^2,$$

$$\text{Lasso: }\frac{1}{2n}\|y-b\mathbf1-Z\beta\|_2^2+\lambda\|\beta\|_1.$$

Both estimators call their penalty **`alpha`** in Python. Report the lasso value as $\lambda$. Numerical penalties are not directly comparable across these differently scaled objectives.

`model__alpha` identifies `alpha` in the pipeline's `model` step. `np.logspace(-2, 6, 25)` creates 25 values from $10^{-2}$ to $10^6$.

Use `scoring="neg_root_mean_squared_error"`. Scikit-learn maximizes scores, so negate them to obtain RMSE in dollars. Useful search results are:

| Attribute | Meaning |
|---|---|
| `best_params_["model__alpha"]` | Chosen penalty |
| `-best_score_` | Mean of the five validation RMSE values |
| `best_estimator_` | Chosen pipeline, refitted on all training rows |

The best penalty is the one with the lowest mean validation RMSE among the values tested.

A larger penalty puts more weight on keeping coefficients small. This can reduce variability but also increase bias. Increasing the penalty does not always improve prediction; the validation curve helps you find a useful balance.


In [9]:
# ridge_search = GridSearchCV(
#     estimator=____,
#     param_grid=____,
#     cv=cv,
#     scoring="neg_root_mean_squared_error",
#     refit=True,
# )
# ridge_search.fit(____, ____)
# print("Ridge alpha:", ridge_search.best_params_["model__alpha"])
# print("Ridge CV RMSE:", -ridge_search.best_score_)

# Repeat the search with lasso_pipeline and lasso_grid.
# Name the fitted search lasso_search.


### 2C. Plot and explain

`cv_results_` stores every tested penalty and its mean validation score. Despite the name `mean_test_score`, these are **validation-fold scores**, not scores on `X2_test`.

Use a logarithmic horizontal axis. If the minimum is at a grid boundary, extend the grid and rerun cross-validation before looking at the test set. Describe whether nearby penalties give similar errors.

Lasso can set slopes to zero; ridge generally shrinks them without making them exactly zero. Use `np.count_nonzero(np.abs(slopes) > 1e-8)` to count the nonzero lasso slopes. A zero slope does not prove that a feature is irrelevant.


In [10]:
# results = pd.DataFrame(ridge_search.cv_results_)
# penalties = results["param_model__alpha"].astype(float)
# rmse = -results["mean_test_score"]
# plt.semilogx(____, ____, marker="o")
# plt.xlabel("Ridge alpha")
# plt.ylabel("Mean validation RMSE (dollars)")
# plt.show()
# Repeat for Lasso with an appropriate axis label.

# slopes = lasso_search.best_estimator_.named_steps["model"].coef_
# number_nonzero = ____


**My ridge choice:**

_Give your grid, chosen alpha, mean validation RMSE, and explanation._

**My lasso choice:**

_Give your grid, chosen lambda, mean validation RMSE, nonzero count, and explanation._

**Connection to the lecture:**

_Why might regularization improve prediction even when OLS is BLUE?_


### Bonus: Elastic net

Elastic net combines the lasso and ridge penalties:

$$\frac{1}{2n}\|y-b\mathbf1-Z\beta\|_2^2
+\alpha\rho\|\beta\|_1+\frac{\alpha(1-\rho)}{2}\|\beta\|_2^2.$$

The overall penalty is `alpha`; the mixing weight $\rho$ is `l1_ratio`. At `l1_ratio=1`, this becomes lasso. Values between zero and one combine both penalties.

Search both parameters together with the same pipeline and CV folds. Report the best tested pair **(`alpha`, `l1_ratio`)**, its mean validation RMSE, and a short explanation. Complete this search before the final test evaluation if you attempt the bonus.


In [11]:
# elastic_pipeline = regression_pipeline(ElasticNet(max_iter=50000, tol=1e-5))
# elastic_grid = {
#     "model__alpha": np.logspace(-2, 4, 13),
#     "model__l1_ratio": [0.2, 0.5, 0.8, 1.0],
# }
# Use the GridSearchCV template above with this pipeline and grid.
# Name the fitted search elastic_search.


**My elastic-net answer:**

_Give the best tested pair, mean validation RMSE, and explanation._


### Final evaluation

Record your selected penalties before using the test set. Each search's `best_estimator_` is already fitted on all training rows.

Predict `X2_test` and compare with `y2_test`. RMSE and MAE are measured in dollars; smaller values indicate smaller errors. $R^2$ compares the predictions with a constant prediction using the test mean; larger is better, and it can be negative.

Evaluate each chosen model once. Use cross-validation, rather than these final test scores, to select penalties. Put the test results in a table with one row per model.


In [12]:
# Write your selected penalties here before evaluating the test set.
# Ridge alpha:
# Lasso lambda:
# Elastic-net pair, if attempted:

# prediction = ridge_search.best_estimator_.predict(____)
# test_rmse = np.sqrt(mean_squared_error(y2_test, ____))
# test_mae = mean_absolute_error(y2_test, ____)
# test_r2 = r2_score(y2_test, ____)
# Repeat for Lasso and, if completed, Elastic Net.
# Display a table with columns: Model, Test RMSE, Test MAE, Test R-squared.


## Before submitting

- **Problem 1:** Include resampling code, both histograms, both kinds of interval for each coefficient, and your comparison.
- **Problem 2:** Include the penalty grids, chosen values, validation RMSE, two validation curves, nonzero lasso count, explanations, and final test table.
- **Bonus, if completed:** Include the elastic-net search and answer.
- Restart the kernel and run your completed notebook from top to bottom.

**Reading:** *Sec4StatisticsML.pdf*, PDF pages 15–17, 20–22, 26–27, and 34–36. Page numbers count from the first PDF page. See **Lab2Master.ipynb** for worked examples and feature descriptions.
